In [6]:
import pandas as pd
import glob
import numpy as np
import statistics as stat
from scipy.stats import skew,kurtosis

In [20]:
class Relatorio:

    def __init__(self, path, header_columns):
        self.diretorio = glob.glob(path)
        self.header_columns = header_columns # quantidade de linhas a serem consideradas como cabeçalhos quando transpostas.
        self.dfs = []

    def _concatenar_dfs(self):
        if not self.dfs:
            return pd.DataFrame()

        return pd.concat(
            self.dfs,
            ignore_index=True
        )

    def _ler_arquivo(self, arquivo, **kwargs):
        try:
            return pd.read_csv(
                arquivo,
                sep=";",
                engine="python",
                **kwargs
            )

        except Exception as erro:
            print(f"Erro em {arquivo}: {erro}")
            return None

    def obter_tabela_header(self):
        self.dfs = []

        for arquivo in self.diretorio:

            df = self._ler_arquivo(
                arquivo,
                nrows=self.header_columns,
                on_bad_lines="warn"
            )

            if df is None or df.empty:
                continue

            # Preservar nome da primeira coluna original
            nome_coluna = df.columns[0]

            # Transpor DataFrame
            df = df.transpose()

            # Definir primeira linha como cabeçalho
            df.columns = df.iloc[0].values

            # Restaurar índice com nome original
            df = df.iloc[1:].reset_index(names=nome_coluna)

            self.dfs.append(df)

            print(f"Header: {arquivo}")

        return self._concatenar_dfs()

    def obter_tabela_data(self):
        self.dfs = []

        for arquivo in self.diretorio:

            df = self._ler_arquivo(
                arquivo,
                header=self.header_columns
            )

            if df is None or df.empty:
                continue

            self.dfs.append(df)

            print(f"Data: {arquivo}")

        return self._concatenar_dfs()

class AnaliseEstatistica:
    def __init__(self,amostra):
        self.amostra = amostra
        self.media = stat.mean(amostra)
        self.moda = stat.mode(amostra)
        self.mediana = stat.median(amostra) 
        self.desvio_padrao = stat.stdev(amostra)
        self.assimetria = skew(amostra)
        self.curtose = kurtosis(amostra)
        self.minimo = min(amostra)
        self.maximo = max(amostra)


if __name__ == "__main__":

    base = Relatorio(
        path=r"dados/raw/*.csv",
        header_columns=6
    )

    amostra=AnaliseEstatistica([4,5,6])

    print(amostra.curtose)


-1.5
